# Từ notebook tới memo: khách vãng lai Capital Bikeshare

Bộ `bike-sharing-uci`, `hour.csv` (17 379 giờ có dữ liệu, 2011-2012). Mỗi dòng là một giờ: `dteday`, `hr` (0-23),
`workingday` (1 = ngày làm việc, 0 = cuối tuần hoặc ngày lễ), `casual` (lượt của khách vãng lai, không có thẻ thành
viên), `registered` (lượt của thành viên), `cnt = casual + registered`.

Giám đốc marketing hỏi: **"Năm sau tôi muốn chạy chiến dịch mời khách vãng lai đăng ký thành viên. Nên tập trung
vào lúc nào?"** Một đồng nghiệp đã làm phân tích trong notebook này (các cell `explore` bên dưới, chạy được ngay).
Việc của bạn là phần một DA senior hay bị thiếu thời gian nhất: **rút ra vài con số trả lời câu hỏi và viết memo
một trang** cho người không đọc notebook.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Cell gắn tag `explore` bị bỏ qua khi chấm.
- Viết `memo_facts(hour)` (phần 2), rồi viết `MEMO.md` cạnh notebook này (phần 3).
- Chạy test: `uv run pytest tracks/data-analyst/modules/15-stakeholder-storytelling-interview/exercises/intermediate/01-memo-from-notebook -q`.
- Xong thì làm phần **trắc nghiệm** trong app.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc hour.csv
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

hour = pd.read_csv(ensure("bike-sharing-uci") / "hour.csv", parse_dates=["dteday"])
print(hour.shape, hour["dteday"].min().date(), "->", hour["dteday"].max().date())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# "Khung giờ chiến dịch": ngày nghỉ (workingday = 0), 10h-17h, tháng 4-10.
WINDOW_MONTHS = range(4, 11)
WINDOW_HOURS = range(10, 18)

## 1. Phân tích của đồng nghiệp (đã làm sẵn)

Ba cell dưới là phần "notebook cho sẵn": khách vãng lai đi khi nào, theo loại ngày, giờ và tháng. Chạy, đọc, và
ghi ra giấy ba điều bạn thấy trước khi đọc phần 2.

In [ ]:
# explore: tỉ trọng khách vãng lai theo loại ngày
day_type = hour["workingday"].map({1: "Ngày làm việc", 0: "Ngày nghỉ"})
by_type = hour.groupby(day_type)[["casual", "registered", "cnt"]].sum()
by_type["casual_share"] = by_type["casual"] / by_type["cnt"]
by_type["share_of_all_casual"] = by_type["casual"] / hour["casual"].sum()
by_type["days"] = hour.groupby(day_type)["dteday"].nunique()
by_type.round(3)

In [ ]:
# explore: lượt TB mỗi giờ, khách vãng lai và thành viên, theo loại ngày
prof = hour.groupby([day_type, "hr"])[["casual", "registered"]].mean().unstack(0)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, who in zip(axes, ["casual", "registered"]):
    prof[who].plot(ax=ax)
    ax.set_title(f"{who}: lượt TB mỗi giờ", loc="left")
    ax.set_xlabel("giờ")
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# explore: khách vãng lai theo tháng (cộng hai năm) - mùa ấm chiếm phần lớn
by_month = hour.groupby(hour["dteday"].dt.month)["casual"].sum()
(by_month / by_month.sum()).round(3).rename("share_of_casual").to_frame().T

## 2. Các con số cho memo

Memo không cần mười bảng; nó cần vài con số trả lời được câu hỏi, mỗi con số có định nghĩa rõ để người khác tính
lại ra đúng số đó. `memo_facts(hour) -> dict` trả **đúng tám khoá** sau (tỉ lệ dạng 0-1, `float`):

| Khoá | Định nghĩa |
| --- | --- |
| `casual_share` | Tổng `casual` / tổng `cnt` |
| `offday_days_share` | Số **ngày** (`dteday` khác nhau) có `workingday = 0` / tổng số ngày |
| `offday_casual_share` | `casual` trong ngày nghỉ / tổng `casual` |
| `casual_share_offday` | Trong ngày nghỉ: tổng `casual` / tổng `cnt` |
| `casual_share_workday` | Trong ngày làm việc: tổng `casual` / tổng `cnt` |
| `window_hours_share` | Số dòng (giờ có dữ liệu) thuộc khung giờ chiến dịch / tổng số dòng |
| `window_casual_share` | `casual` trong khung / tổng `casual` |
| `casual_share_in_window` | Trong khung: tổng `casual` / tổng `cnt` |

Khung giờ chiến dịch: `workingday = 0`, `hr` thuộc `WINDOW_HOURS` (10-17), tháng của `dteday` thuộc
`WINDOW_MONTHS` (4-10). `dteday` có thể là chuỗi hoặc datetime; không sửa `hour` đầu vào.

In [ ]:
def memo_facts(hour: pd.DataFrame) -> dict:
    """Tám con số cho memo (xem bảng định nghĩa ở trên)."""
    # TODO: df = hour.copy(); df["dteday"] = pd.to_datetime(df["dteday"])
    # TODO: tách ngày nghỉ / ngày làm việc; mặt nạ khung giờ chiến dịch
    # TODO: trả dict tám khoá, giá trị float
    raise NotImplementedError

In [ ]:
# explore: các con số, định dạng sẵn để chép vào memo
facts = memo_facts(hour)
for k, v in facts.items():
    print(f"{k:24s} {v:.1%}")

## 3. Viết memo

Mở `MEMO.md` (cùng thư mục với notebook) và viết memo một trang cho giám đốc marketing, theo khung có sẵn:

- **Dòng đầu** `# ...` là headline: kết luận có ít nhất một con số lấy từ `memo_facts` (phần trăm, làm tròn tới số
  nguyên hoặc một chữ số thập phân, ví dụ `28%` hoặc `27.6%`).
- Năm mục theo thứ tự: `## Bối cảnh và câu hỏi`, `## Khuyến nghị`, `## Bằng chứng` (ít nhất ba gạch đầu dòng, mỗi
  dòng một con số), `## Giới hạn`, `## Bước tiếp theo`.
- 200-450 từ. Viết cho người không đọc code: không tên cột (`workingday`), không tên tệp, không thuật ngữ pandas.

Câu hỏi tự kiểm trước khi nộp:

1. Người đọc dừng sau dòng đầu có biết nên làm gì không?
2. Mỗi con số có tham chiếu ("so với...", "trong khi chỉ chiếm...") không?
3. Dữ liệu **không** nói gì? Nó cho biết khách vãng lai đi khi nào, không cho biết ai trong số họ sẽ đăng ký, cũng
   không có vị trí trạm. Mục giới hạn và bước tiếp theo là chỗ nói điều đó.